# MarketLens on Kaggle
Enable GPU and Internet. Upload the product source as a Kaggle Dataset or clone your published product repository. This notebook imports the package; it does not duplicate pipeline logic. Live execution is not yet validated. Store TAVILY_API_KEY in Kaggle Secrets. Download files from /kaggle/working before ending the session.


In [ ]:
from pathlib import Path

# Set this to the uploaded product source directory.
PROJECT = Path('/kaggle/input/marketlens-source/project')
assert (PROJECT / 'pyproject.toml').exists(), 'Attach the product source dataset and update PROJECT'


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', str(PROJECT) + '[research]'], check=True)


In [ ]:
import os

import torch
from kaggle_secrets import UserSecretsClient

os.environ['TAVILY_API_KEY'] = UserSecretsClient().get_secret('TAVILY_API_KEY')
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
print(torch.cuda.get_device_name(), torch.cuda.get_device_properties(0).total_memory // 2**20, 'MiB')


In [ ]:
from marketlens.models import ResearchRequest, Settings
from marketlens.pipeline import run_research

request = ResearchRequest.model_validate_json((PROJECT / 'examples/request.json').read_text())
bundle = run_research(request, Settings(), '/kaggle/working/marketlens')
print(bundle.status, bundle.manifest)


In [ ]:
import shutil

shutil.make_archive('/kaggle/working/marketlens-bundle', 'zip', '/kaggle/working/marketlens')


## Acceptance
Run again with the same output path to test resume. Review citation integrity and at least 30 claims manually. Build real-source labels following docs/evaluation.md, then run all three retrieval modes on the same held-out corpus. A completed synthetic test is not a live model benchmark.
